# Fuzzy Factory · Reproducible commerce analytics

An original AI-assisted implementation based on the project topics in the supplied notebook. Official CSV records are included unchanged. The completed dashboard needs no notebook execution. This notebook is an optional technical walkthrough.


In [1]:
from pathlib import Path
import duckdb
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
con=duckdb.connect(str(ROOT/'data/warehouse/fuzzy.duckdb'),read_only=True)


## Source profile

Verify full source counts and time coverage. Source boundary months are partial.


In [2]:
tables=['orders','order_items','order_item_refunds','products','website_sessions','website_pageviews']
{t:con.execute(f'select count(*) from raw.{t}').fetchone()[0] for t in tables}


{'orders': 32313,
 'order_items': 40025,
 'order_item_refunds': 1731,
 'products': 4,
 'website_sessions': 472871,
 'website_pageviews': 1188124}

In [3]:
con.execute('select min(session_date) as start_date,max(session_date) as end_date from fct_sessions').fetchdf()


,start_date,end_date
0,2012-03-19,2015-03-19


## Correct grain first

Item/refund joins must not multiply revenue. Refunds are pre-aggregated before item, order and session rollups.


In [4]:
con.execute('select (select sum(price_usd) from raw.orders) as source_sales,(select sum(revenue_usd) from fct_order_items) as item_sales,(select sum(revenue_usd) from fct_orders) as order_sales').fetchdf()


,source_sales,item_sales,order_sales
0,1938509.75,1938509.75,1938509.75


## Executive


In [5]:
con.execute((ROOT/'sql/01_executive.sql').read_text()).fetchdf()


,orders,buyers,sales,cogs,gross_profit,refunds,net_sales,aov
0,32313,31696,1938509.75,722370.25,1216139.5,85338.69,1853171.06,59.991636


## Monthly Growth


In [6]:
con.execute((ROOT/'sql/02_monthly_growth.sql').read_text()).fetchdf()


,month,orders,sales,gross_profit,complete_month,previous_sales,previous_complete,sales_mom
0,2012-03-01,60,2999.40,1830.0,False,NaN,<NA>,NaN
1,2012-04-01,99,4949.01,3019.5,True,2999.40,False,NaN
2,2012-05-01,108,5398.92,3294.0,True,4949.01,True,0.090909
3,2012-06-01,140,6998.60,4270.0,True,5398.92,True,0.296296
4,2012-07-01,169,8448.31,5154.5,True,6998.60,True,0.207143
5,2012-08-01,228,11397.72,6954.0,True,8448.31,True,0.349112
6,2012-09-01,287,14347.13,8753.5,True,11397.72,True,0.258772
7,2012-10-01,371,18546.29,11315.5,True,14347.13,True,0.292683
8,2012-11-01,618,30893.82,18849.0,True,18546.29,True,0.665768
9,2012-12-01,506,25294.94,15433.0,True,30893.82,True,-0.181230


## Campaigns


In [7]:
con.execute((ROOT/'sql/03_campaigns.sql').read_text()).fetchdf()


,channel,campaign,device_type,sessions,converted,conversion,sales,sales_per_session
0,gsearch,nonbrand,desktop,195155,16037.0,0.082176,956016.90,4.898757
1,gsearch,nonbrand,mobile,87551,2785.0,0.031810,168397.31,1.923420
2,bsearch,nonbrand,desktop,47395,3584.0,0.075620,211434.40,4.461112
3,Organic / referral,Unattributed,desktop,28292,2818.0,0.099604,170086.70,6.011830
4,Direct,Unattributed,desktop,23284,2284.0,0.098093,140098.91,6.016961
5,gsearch,brand,desktop,19922,2100.0,0.105411,126801.76,6.364911
6,Direct,Unattributed,mobile,16633,572.0,0.034389,34524.93,2.075689
7,Organic / referral,Unattributed,mobile,15119,444.0,0.029367,26722.49,1.767477
8,gsearch,brand,mobile,13407,411.0,0.030656,24928.92,1.859396
9,bsearch,nonbrand,mobile,7514,234.0,0.031142,14129.08,1.880367


## Products


In [8]:
con.execute((ROOT/'sql/04_products.sql').read_text()).fetchdf()


,product_name,items,sales,gross_profit,refunds,refunded_item_rate
0,The Original Mr. Fuzzy,24226,1211057.74,738893.0,61837.63,0.051061
1,The Forever Love Bear,5796,347702.04,217350.0,7738.71,0.022257
2,The Birthday Sugar Panda,4985,229260.15,157027.5,13842.99,0.060381
3,The Hudson River Mini bear,5018,150489.82,102869.0,1919.36,0.012754


## Ordered Funnel


In [9]:
con.execute((ROOT/'sql/05_ordered_funnel.sql').read_text()).fetchdf()


,sessions,products,cart,shipping,billing,completed
0,472871,261231.0,94953.0,64484.0,52058.0,32313.0


## Repeat Buyers


In [10]:
con.execute((ROOT/'sql/06_repeat_buyers.sql').read_text()).fetchdf()


,buyers,repeat_buyers,repeat_rate
0,31696,591,0.018646


## Purchase Cohorts


In [11]:
con.execute((ROOT/'sql/07_purchase_cohorts.sql').read_text()).fetchdf()


,cohort,months_since,active_buyers,cohort_buyers,purchase_retention
0,2012-03-01,0,60,60,1.000000
1,2012-04-01,0,99,99,1.000000
2,2012-05-01,0,108,108,1.000000
3,2012-06-01,0,139,139,1.000000
4,2012-06-01,1,1,139,0.007194
...,...,...,...,...,...
112,2015-01-01,1,30,2049,0.014641
113,2015-01-01,2,15,2049,0.007321
114,2015-02-01,0,2004,2004,1.000000
115,2015-02-01,1,16,2004,0.007984


## Quarterly Patterns


In [12]:
con.execute((ROOT/'sql/08_quarterly_patterns.sql').read_text()).fetchdf()


,year,quarter,sessions,orders,conversion,sales_per_order,sales_per_session
0,2012,1,1879,60.0,0.031932,49.990000,1.596275
1,2012,2,11433,347.0,0.030351,49.990000,1.517233
2,2012,3,16892,684.0,0.040493,49.990000,2.024222
3,2012,4,32266,1495.0,0.046334,49.990000,2.316217
4,2013,1,19833,1273.0,0.064186,52.142396,3.346809
5,2013,2,24745,1718.0,0.069428,51.538312,3.578211
6,2013,3,27663,1840.0,0.066515,51.734533,3.441114
7,2013,4,40540,2616.0,0.064529,54.715688,3.530741
8,2014,1,46779,3069.0,0.065606,62.160684,4.078136
9,2014,2,53129,3848.0,0.072427,64.374207,4.662462


## Frequency Recency


In [13]:
con.execute((ROOT/'sql/09_frequency_recency.sql').read_text()).fetchdf()


,user_id,days_since_purchase,orders,observed_sales,frequency
0,341972,1,3,251.94,3+ orders
1,324814,1,3,245.95,3+ orders
2,172266,310,3,237.95,3+ orders
3,275098,125,3,235.95,3+ orders
4,142410,398,3,219.96,3+ orders
...,...,...,...,...,...
31691,364829,44,1,29.99,1 order
31692,380396,22,1,29.99,1 order
31693,383269,17,1,29.99,1 order
31694,391109,6,1,29.99,1 order


## Page Popularity


In [14]:
con.execute((ROOT/'sql/10_page_popularity.sql').read_text()).fetchdf()


,pageview_url,pageviews,sessions
0,/products,261231,261231
1,/the-original-mr-fuzzy,162525,162525
2,/home,137576,137576
3,/lander-2,131170,131170
4,/cart,94953,94953
5,/lander-3,79000,79000
6,/lander-5,68166,68166
7,/shipping,64484,64484
8,/billing-2,48441,48441
9,/lander-1,47574,47574


## Landing Focus


In [15]:
con.execute((ROOT/'sql/11_landing_focus.sql').read_text()).fetchdf()


,landing_page,sessions,bounce_rate,reached_products,completed_ordered_path,product_to_complete,session_conversion
0,/home,3038,0.534562,1414.0,118.0,0.083451,0.038841
1,/lander-1,2350,0.531489,1101.0,94.0,0.085377,0.040000


## Interpretation limits

Gross profit is not net profit. Conversion does not establish ROI without ad spend. Refunds are allocated to original sale date. Cohorts use first purchase across full history; future cells are blank. Landing-page differences are descriptive, not an automatically valid randomized experiment. Add your own recommendations after reviewing the dashboard and SQL.


In [16]:
con.close()
